# 🧠 LeetCode 76: Minimum Window Substring

**Pattern:** Hard Sliding Window (Two Hash Maps)

- **Created:** 2026-02-28
- **Focus:** Tracking 'have' vs 'need' conditions to shrink a valid window
- **Tags:** `string` | `sliding-window` | `hash-table` | `hard` | `citi-prep`

## 📖 Problem Statement

Given two strings `s` and `t` of lengths `m` and `n` respectively, return the **minimum window substring** of `s` such that every character in `t` (including duplicates) is included in the window.

If there is no such substring, return the empty string `""`.

### Example 1:
- **Input:** `s = "ADOBECODEBANC"`, `t = "ABC"`
- **Output:** `"BANC"`
- **Explanation:** The minimum window substring "BANC" includes 'A', 'B', and 'C' from string t.

## 🧠 Mental Model & Intuition

This is the quintessential "Hard" Sliding Window pattern. It has two phases:

### Phase 1: Expand until Valid (The Hunt)
Stretch the right side of your frame until you have captured all the letters required by string `t`. To know when you've succeeded, you don't check the whole dictionary every step. You keep two integer counters: `have` and `need`. `need` is the number of distinct characters in `t`. `have` increases every time your window captures the exact required amount of a specific character.

### Phase 2: Shrink until Invalid (The Optimization)
Once `have == need`, your window is valid! But it might be too big (e.g., "ADOBEC" contains "ABC", but is 6 letters long). Now, crunch the left side of the frame forward. Keep recording the minimum size observed. Stop shrinking the moment you accidentally throw out a letter you needed (when `have` drops below `need`).

Then, resume Phase 1 to go hunting again.

## 🐢 Brute Force Approach

Check every possible substring. Count the frequencies of its characters and see if it satisfies the requirement of `t`.

```python
def minWindowBrute(s, t):
    if t == "": return ""
    from collections import Counter
    t_count = Counter(t)
    min_str = "" 
    for i in range(len(s)):
        for j in range(i + 1, len(s) + 1):
            sub = s[i:j]
            sub_count = Counter(sub)
            valid = True
            for char in t_count:
                if sub_count[char] < t_count[char]:
                    valid = False
                    break
            if valid:
                if min_str == "" or len(sub) < len(min_str):
                    min_str = sub
    return min_str
```
**Time:** $O(N^3)$ (Massive Timeout) | **Space:** $O(N)$

In [ ]:
# Optimal Sliding Window Approach
def minWindow(s: str, t: str) -> str:
    if t == "": return ""
    
    countT = {} # What we need
    window = {} # What we have in our current frame
    
    for char in t:
        countT[char] = countT.get(char, 0) + 1
        
    have, need = 0, len(countT)
    res = [-1, -1] # [Left index, Right index]
    resLen = float("infinity")
    
    l = 0
    for r in range(len(s)):
        c = s[r]
        window[c] = window.get(c, 0) + 1
        
        # If the newly added char is one we care about, and we just hit the exact number we needed
        if c in countT and window[c] == countT[c]:
            have += 1
            
        # Phase 2: Shrink from the left while the window remains valid
        while have == need:
            # 1. First, check if this valid window is a new record
            if (r - l + 1) < resLen:
                res = [l, r]
                resLen = (r - l + 1)
                
            # 2. Pop the left character from our window
            left_char = s[l]
            window[left_char] -= 1
            
            # 3. If the character we just popped was important, did it break our valid state?
            if left_char in countT and window[left_char] < countT[left_char]:
                have -= 1
                
            # 4. Advance left pointer
            l += 1
            
    left, right = res
    return s[left : right + 1] if resLen != float("infinity") else ""

print("Min window for ADOBECODEBANC: ", minWindow("ADOBECODEBANC", "ABC"))

## ⏱️ Complexity Analysis

- **Time Complexity:** $O(|S| + |T|)$. We iterate through `t` once to build the requirement map. Then `s` is traversed. Even with the inner `while` loop, the `l` pointer only moves forward. Every character in `s` is added to the window once, and removed from the window at most once. 
- **Space Complexity:** $O(|S| + |T|)$ or technically $O(1)$ if assuming ASCII. The maps store character counts, which cannot exceed the alphabet size.

## 🎤 Interview Q&A

### Q1: Why do `have` and `need` track the *number of distinct characters fulfilled* rather than the total letter count?
**Answer:** Because it avoids an $O(26)$ verification loop inside the window loop. If `t` is "AABC", we need 2 A's, 1 B, and 1 C. If we just tracked `total_have`, 4 A's would incorrectly signal success. By tracking distinct conditions met (`have=3` means A's quota is met, B's quota is met, C's quota is met), evaluating validity is a simple integer comparison `have == need` which is $O(1)$.

---

### Q2: Why does the code check `window[c] == countT[c]` and not greater-than-or-equal?
**Answer:** If we need 2 A's, the moment we capture the 2nd 'A', we increment `have`. If we capture a 3rd 'A', we don't increment `have` again because that specific character quota was already fulfilled. Checking strictly for equality precisely isolates the moment the condition flips from unfulfilled to fulfilled.

## 📚 Key Terminology

| Term | Definition | Example |
|---|---|---|
| **Valid Window** | A window state that fully satisfies the problem's criteria. | `have == need` |
| **Two Pointers (L/R)** | Bounding indices defining the active evaluation space. | `s[l:r]` |
| **Condition Satisfaction Tracker** | Using an integer pair to bypass dictionary-equality checks. | `have += 1` |

## 💼 The Citi Narrative

**Context:** Finding full sequences in corrupted application logs.

**Scenario:** A trading application occasionally spat out fragmented log files where standard transaction flows (Step A -> Step B -> Step C) were interleaved with thousands of garbage network warnings. We needed to extract the tightest bound "clean" log block that contained all required steps for an audit, allowing compliance officers to read the minimal amount of surrounding context.

**Impact:** A junior dev's script crashed the log aggregator because it tried to run Regex over a 50GB text stream attempting to match disparate events. Replacing it with this Sliding Window algorithm in C# converted a multi-hour batch regex process into a single-pass streaming parser that identified the minimal audit blocks in seconds with negligible memory footprint.

In [ ]:
# EXERCISE: Trace the Shrink phase
# s = 'ADOBEC', t='ABC'. 
# You just added C. have == need.
# L is at 0 ('A'). You pop 'A'. have goes down to 2.
# The window is now invalid. R must now advance to find another 'A'.

## 🎯 Summary: Key Takeaways

### The Pattern
**Hard Sliding Window** — Tracking a variable `have` vs `need` state to dynamically expand and aggressively shrink.

### When to Use It
- ✅ Extracting tightest valid sequences containing required keywords in logs.
- ✅ Satisfying complex subset requirements in a continuous data stream.
- ❌ **Don't use when:** Only looking for exact contiguous match of the entire subset exactly.

### Complexity
| Approach | Time | Space |
|----------|------|-------|
| Brute Force | $O(N^3)$ | $O(S + T)$ |
| Optimal | $O(S + T)$ | $O(S + T)$ |

### Interview Confidence Checklist
- [ ] Can explain the brute force and why it fails
- [ ] Can state the pattern name and core insight in one sentence
- [ ] Can write the optimal solution from memory
- [ ] Can state time and space complexity with justification
- [ ] Can name at least one real-world / Citi application

---

**"Simplicity and clarity is Gold."** — Sean's Study Mantra

Master **Hard Sliding Window** and you've mastered one of the most common patterns in data engineering interviews. 🚀